In [0]:
# install and verify libraries

#1. installing library

%pip install beautifulsoup4


#2 verify installation

try:
    from bs4 import BeautifulSoup
    print("BeautifulSoup installed successfully and running")
except ImportError:
    print("Library not installed")

In [0]:
# Setup and dynamic link finder. Will search for Download button

import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta

# 1. define url

base_page_url = (
    "https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open"
)

# 2. Azure storage paths

storage_account_name = "REMOVED"
storage_account_key = "REMOVED"

# Connect to ADLS

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# AUTO create bronze container if missing
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# define folders

landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions2/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions2/archive/"



# 3. Dynamic link folder

def get_dyncamic_download_url():
    print(f"Scrapping{base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(base_page_url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    # Logic: Find link containing 'download' and 'csv'
    for link in soup.find_all('a',href=True):
        href = link['href']
        if "download" in href and "transactions.csv" in href:
            final_url = f"https://www.dubaipulse.gov.ae{href}" if href.startswith("/") else href
            print(f"Found download link: {final_url}")
            return final_url
    raise Exception("Download link not found")
print("configuration loaded and Auto-create enabled")




 

In [0]:
# 3. Retention policy
from datetime import datetime, timedelta
def cleanup_old_files(days_to_keep=14):
    print("Checking archive for old files")
    cutoff_date=datetime.now() - timedelta(days=days_to_keep)
    try :
        files = dbutils.fs.ls(archive_zone)
        for f in files:
            filename = f.name.replace(".csv","")
            try :
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date  < cutoff_date :
                    print(f"Deleting : {f.name}")
                    dbutils.fs.rm(f.path)
            except ValueError:
                continue
    except Exception:
        print("No files in archive")

cleanup_old_files()


In [0]:
# 4. Archival logic
from datetime import datetime, timedelta

def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"
    try:
        dbutils.fs.ls(current_file_path)
        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"

        print("Archiving from Landing to Archive path")
        dbutils.fs.mv(current_file_path,archive_path)
    except Exception:
        print("No current.csv file found in Landing path")

archive_current_file()
    


In [0]:
# 5. Download current snapshot

import os
import time


def download_snapshot_with_retry(max_retries=3):
    fresh_url = get_dyncamic_download_url()
    local_path = "/tmp/current.csv"
    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"attempt {attempt} of {max_retries} for downloading to driver node..."
            )
            with request.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(local_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=10 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                print("download completed")
                break
        except Exception as e:
            print(f"attempt {attempt} failed : {str(e)}")
            if attempt < max_retries:
                print("waiting for 5 secs...")
                time.sleep(5)
            else:
                print("All retires failed...")
                raise e
        if os.path.exists(local_path):
            print("Uploading to Bronze landing Zone")
            dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
            os.remove(local_path)
            print("Successful update of current.csv file in Deta lake.")
            break
        else:
            print("No file downloaded. Retrying...")


#download_snapshot_with_retry()